In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
project_path = "/content/drive/MyDrive/Mobile_API_Misuse_Detector/"

In [8]:
log_file = project_path + "data/raw/access.log"

with open(log_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(lines[:5])

['54.36.149.41 - - [22/Jan/2019:03:56:14 +0330] "GET /filter/27|13%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C%DA%A9%D8%B3%D9%84,27|%DA%A9%D9%85%D8%AA%D8%B1%20%D8%A7%D8%B2%205%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C%DA%A9%D8%B3%D9%84,p53 HTTP/1.1" 200 30577 "-" "Mozilla/5.0 (compatible; AhrefsBot/6.1; +http://ahrefs.com/robot/)" "-"\n', '31.56.96.51 - - [22/Jan/2019:03:56:16 +0330] "GET /image/60844/productModel/200x200 HTTP/1.1" 200 5667 "https://www.zanbil.ir/m/filter/b113" "Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build/HuaweiALE-L21) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/66.0.3359.158 Mobile Safari/537.36" "-"\n', '31.56.96.51 - - [22/Jan/2019:03:56:16 +0330] "GET /image/61474/productModel/200x200 HTTP/1.1" 200 5379 "https://www.zanbil.ir/m/filter/b113" "Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build/HuaweiALE-L21) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/66.0.3359.158 Mobile Safari/537.36" "-"\n', '40.77.167.129 - - [22/Jan/2019:03:56:17 +0330] "GET /image/14925/productModel/100x100 HT

In [9]:
import re


In [10]:
log_pattern = re.compile(
    r'(\d+\.\d+\.\d+\.\d+) .*? '
    r'\[(.*?)\] '
    r'"(GET|POST) (.*?) HTTP.*?" '
    r'(\d+) .*? '
    r'"(.*?)" '
    r'"(.*?)"'
)

In [7]:
match = log_pattern.search(lines[0])

print(match.groups())

('54.36.149.41', '22/Jan/2019:03:56:14 +0330', 'GET', '/filter/27|13%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C%DA%A9%D8%B3%D9%84,27|%DA%A9%D9%85%D8%AA%D8%B1%20%D8%A7%D8%B2%205%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C%DA%A9%D8%B3%D9%84,p53', '200', '-', 'Mozilla/5.0 (compatible; AhrefsBot/6.1; +http://ahrefs.com/robot/)')


In [11]:
import re
import pandas as pd

parsed_logs = []
max_lines = 50000   # commence avec 50k lignes seulement

with open(log_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):

        if i >= max_lines:
            break

        match = log_pattern.search(line)

        if match:
            parsed_logs.append({
                "ip": match.group(1),
                "timestamp": match.group(2),
                "method": match.group(3),
                "endpoint": match.group(4),
                "status_code": int(match.group(5)),
                "referer": match.group(6),
                "user_agent": match.group(7)
            })

df = pd.DataFrame(parsed_logs)

print(df.shape)
df.head()

(49456, 7)


,ip,timestamp,method,endpoint,status_code,referer,user_agent
0,54.36.149.41,22/Jan/2019:03:56:14 +0330,GET,/filter/27|13%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C...,200,-,Mozilla/5.0 (compatible; AhrefsBot/6.1; +http:...
1,31.56.96.51,22/Jan/2019:03:56:16 +0330,GET,/image/60844/productModel/200x200,200,https://www.zanbil.ir/m/filter/b113,Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build...
2,31.56.96.51,22/Jan/2019:03:56:16 +0330,GET,/image/61474/productModel/200x200,200,https://www.zanbil.ir/m/filter/b113,Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build...
3,40.77.167.129,22/Jan/2019:03:56:17 +0330,GET,/image/14925/productModel/100x100,200,-,Mozilla/5.0 (compatible; bingbot/2.0; +http://...
4,91.99.72.15,22/Jan/2019:03:56:17 +0330,GET,/product/31893/62100/%D8%B3%D8%B4%D9%88%D8%A7%...,200,-,Mozilla/5.0 (Windows NT 6.2; Win64; x64; rv:16...


In [12]:
df["is_mobile"] = df["user_agent"].str.contains(
    "Android|iPhone|Mobile|Dalvik",
    case=False,
    na=False
).astype(int)

df["is_bot"] = df["user_agent"].str.contains(
    "bot|crawler|spider|Googlebot|bingbot|AhrefsBot",
    case=False,
    na=False
).astype(int)

df["is_error"] = (df["status_code"] >= 400).astype(int)

df["is_post"] = (df["method"] == "POST").astype(int)

df.head()

,ip,timestamp,method,endpoint,status_code,referer,user_agent,is_mobile,is_bot,is_error,is_post
0,54.36.149.41,22/Jan/2019:03:56:14 +0330,GET,/filter/27|13%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C...,200,-,Mozilla/5.0 (compatible; AhrefsBot/6.1; +http:...,0,1,0,0
1,31.56.96.51,22/Jan/2019:03:56:16 +0330,GET,/image/60844/productModel/200x200,200,https://www.zanbil.ir/m/filter/b113,Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build...,1,0,0,0
2,31.56.96.51,22/Jan/2019:03:56:16 +0330,GET,/image/61474/productModel/200x200,200,https://www.zanbil.ir/m/filter/b113,Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build...,1,0,0,0
3,40.77.167.129,22/Jan/2019:03:56:17 +0330,GET,/image/14925/productModel/100x100,200,-,Mozilla/5.0 (compatible; bingbot/2.0; +http://...,0,1,0,0
4,91.99.72.15,22/Jan/2019:03:56:17 +0330,GET,/product/31893/62100/%D8%B3%D8%B4%D9%88%D8%A7%...,200,-,Mozilla/5.0 (Windows NT 6.2; Win64; x64; rv:16...,0,0,0,0


In [13]:
df.to_csv(
    project_path + "data/processed/parsed_logs_sample.csv",
    index=False
)

In [14]:
ip_request_count = df.groupby("ip").size()

print(ip_request_count.head())

ip
10.124.2.223       2
102.102.124.56     1
103.102.221.226    1
103.216.160.242    2
103.28.132.72      1
dtype: int64


In [15]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="%d/%b/%Y:%H:%M:%S %z"
)

df["minute"] = df["timestamp"].dt.floor("min")

In [16]:
req_per_min_df = df.groupby(["ip", "minute"]).size().reset_index(name="req_per_min")

avg_req_per_min = req_per_min_df.groupby("ip")["req_per_min"].mean().reset_index()
avg_req_per_min.columns = ["ip", "avg_req_per_min"]

max_req_per_min = req_per_min_df.groupby("ip")["req_per_min"].max().reset_index()
max_req_per_min.columns = ["ip", "max_req_per_min"]

In [17]:
req_per_min_df.head(20)

,ip,minute,req_per_min
0,10.124.2.223,2019-01-22 06:13:00+03:30,2
1,102.102.124.56,2019-01-22 06:11:00+03:30,1
2,103.102.221.226,2019-01-22 04:56:00+03:30,1
3,103.216.160.242,2019-01-22 05:22:00+03:30,2
4,103.28.132.72,2019-01-22 06:59:00+03:30,1
5,104.129.192.119,2019-01-22 05:02:00+03:30,2
6,104.129.192.119,2019-01-22 06:45:00+03:30,1
7,104.129.192.119,2019-01-22 06:46:00+03:30,1
8,104.156.210.196,2019-01-22 04:20:00+03:30,1
9,104.194.24.147,2019-01-22 05:19:00+03:30,5


In [18]:
df["window_5min"] = df["timestamp"].dt.floor("5min")

In [19]:
req_count_5min_df = df.groupby(
    ["ip", "window_5min"]
).size().reset_index(name="req_count_5min")

error_rate_5min_df = df.groupby(
    ["ip", "window_5min"]
)["is_error"].mean().reset_index(name="error_rate_5min")

error_analysis_df = error_rate_5min_df.merge(
    req_count_5min_df,
    on=["ip", "window_5min"]
)

suspicious_errors = error_analysis_df[
    (error_analysis_df["req_count_5min"] >= 5) &
    (error_analysis_df["error_rate_5min"] >= 0.5)
]

suspicious_errors.sort_values(
    by=["error_rate_5min", "req_count_5min"],
    ascending=False
).head(20)

,ip,window_5min,error_rate_5min,req_count_5min
804,31.184.130.52,2019-01-22 04:10:00+03:30,1.000000,46
1037,37.9.113.152,2019-01-22 05:00:00+03:30,1.000000,8
1047,37.9.113.152,2019-01-22 06:10:00+03:30,1.000000,6
1035,37.9.113.152,2019-01-22 04:50:00+03:30,1.000000,5
1042,37.9.113.152,2019-01-22 05:25:00+03:30,1.000000,5
1034,37.9.113.152,2019-01-22 04:45:00+03:30,0.833333,6
1045,37.9.113.152,2019-01-22 05:40:00+03:30,0.833333,6
37,104.36.19.231,2019-01-22 04:00:00+03:30,0.727273,11
247,158.58.52.65,2019-01-22 06:55:00+03:30,0.727273,11
360,174.233.0.60,2019-01-22 04:35:00+03:30,0.727273,11


In [20]:
# =====================================================
# 1. request_count
# =====================================================
request_count_df = df.groupby("ip").size().reset_index(name="request_count")


# =====================================================
# 2. max_req_count_5min + avg_req_count_5min
# =====================================================
max_req_count_5min_df = req_count_5min_df.groupby("ip")["req_count_5min"].max().reset_index(
    name="max_req_count_5min"
)

avg_req_count_5min_df = req_count_5min_df.groupby("ip")["req_count_5min"].mean().reset_index(
    name="avg_req_count_5min"
)


# =====================================================
# 3. requests_per_second
# =====================================================
time_span_df = df.groupby("ip")["timestamp"].agg(["min", "max"]).reset_index()
time_span_df["duration_seconds"] = (
    time_span_df["max"] - time_span_df["min"]
).dt.total_seconds()

time_span_df["duration_seconds"] = time_span_df["duration_seconds"].replace(0, 1)

requests_per_second_df = request_count_df.merge(
    time_span_df[["ip", "duration_seconds"]],
    on="ip",
    how="left"
)

requests_per_second_df["requests_per_second"] = (
    requests_per_second_df["request_count"] / requests_per_second_df["duration_seconds"]
)

requests_per_second_df = requests_per_second_df[["ip", "requests_per_second"]]


# =====================================================
# 4. avg_time_between_requests
# =====================================================
df_sorted = df.sort_values(["ip", "timestamp"]).copy()

df_sorted["time_diff_seconds"] = df_sorted.groupby("ip")["timestamp"].diff().dt.total_seconds()

avg_time_between_requests_df = df_sorted.groupby("ip")["time_diff_seconds"].mean().reset_index(
    name="avg_time_between_requests"
)

avg_time_between_requests_df["avg_time_between_requests"] = avg_time_between_requests_df[
    "avg_time_between_requests"
].fillna(0)


# =====================================================
# 5. unique_endpoints
# =====================================================
unique_endpoints_df = df.groupby("ip")["endpoint"].nunique().reset_index(
    name="unique_endpoints"
)


# =====================================================
# 6. unique_ids_accessed
# =====================================================
df["ids_in_endpoint"] = df["endpoint"].str.findall(r"\d+")

unique_ids_df = df.groupby("ip")["ids_in_endpoint"].apply(
    lambda lists: len(set([item for sublist in lists for item in sublist]))
).reset_index(name="unique_ids_accessed")


# =====================================================
# 7. status_404_count
# =====================================================
df["is_404"] = (df["status_code"] == 404).astype(int)

status_404_df = df.groupby("ip")["is_404"].sum().reset_index(
    name="status_404_count"
)


# =====================================================
# 8. max_error_rate_5min
# =====================================================
max_error_rate_5min_df = error_analysis_df.groupby("ip")["error_rate_5min"].max().reset_index(
    name="max_error_rate_5min"
)


# =====================================================
# 9. suspicious_ua_ratio
# =====================================================
suspicious_patterns = (
    "sqlmap|curl|python|requests|bot|crawler|spider|scanner|wget|nikto|java"
)

df["suspicious_user_agent"] = df["user_agent"].str.contains(
    suspicious_patterns,
    case=False,
    na=False
).astype(int)

suspicious_ua_df = df.groupby("ip")["suspicious_user_agent"].mean().reset_index(
    name="suspicious_ua_ratio"
)


# =====================================================
# 10. bot_ratio
# =====================================================
bot_ratio_df = df.groupby("ip")["is_bot"].mean().reset_index(
    name="bot_ratio"
)


# =====================================================
# 11. mobile_ratio
# =====================================================
mobile_ratio_df = df.groupby("ip")["is_mobile"].mean().reset_index(
    name="mobile_ratio"
)


# =====================================================
# 12. post_frequency
# =====================================================
post_frequency_df = df.groupby("ip")["is_post"].mean().reset_index(
    name="post_frequency"
)


# =====================================================
# 13. max_repeated_endpoint_hits
# =====================================================
endpoint_hits_df = df.groupby(["ip", "endpoint"]).size().reset_index(name="endpoint_hits")

max_repeated_endpoint_hits_df = endpoint_hits_df.groupby("ip")["endpoint_hits"].max().reset_index(
    name="max_repeated_endpoint_hits"
)
# =====================================================
# Brute Force Features
# =====================================================

login_patterns = "/login|/auth|/signin|/account/login|/user/login"

df["is_login_attempt"] = df["endpoint"].str.contains(
    login_patterns,
    case=False,
    na=False
).astype(int)

df["is_failed_login"] = (
    (df["is_login_attempt"] == 1) &
    (df["status_code"].isin([401, 403]))
).astype(int)


# 1. login_attempt_count
login_attempt_count_df = df.groupby("ip")["is_login_attempt"].sum().reset_index(
    name="login_attempt_count"
)


# 2. failed_login_count
failed_login_count_df = df.groupby("ip")["is_failed_login"].sum().reset_index(
    name="failed_login_count"
)


# 3. failed_login_rate
failed_login_rate_df = login_attempt_count_df.merge(
    failed_login_count_df,
    on="ip",
    how="left"
)

failed_login_rate_df["failed_login_rate"] = (
    failed_login_rate_df["failed_login_count"] /
    failed_login_rate_df["login_attempt_count"].replace(0, 1)
)

failed_login_rate_df = failed_login_rate_df[["ip", "failed_login_rate"]]


# 4. login_requests_per_min
login_df = df[df["is_login_attempt"] == 1].copy()

if len(login_df) > 0:
    login_req_per_min_df = login_df.groupby(
        ["ip", "minute"]
    ).size().reset_index(name="login_req_per_min")

    max_login_req_per_min_df = login_req_per_min_df.groupby(
        "ip"
    )["login_req_per_min"].max().reset_index(
        name="max_login_req_per_min"
    )
else:
    max_login_req_per_min_df = pd.DataFrame(
        columns=["ip", "max_login_req_per_min"]
    )


# 5. avg_time_between_login_attempts
if len(login_df) > 0:
    login_sorted = login_df.sort_values(["ip", "timestamp"]).copy()

    login_sorted["login_time_diff_seconds"] = login_sorted.groupby(
        "ip"
    )["timestamp"].diff().dt.total_seconds()

    avg_time_between_login_attempts_df = login_sorted.groupby(
        "ip"
    )["login_time_diff_seconds"].mean().reset_index(
        name="avg_time_between_login_attempts"
    )

    avg_time_between_login_attempts_df["avg_time_between_login_attempts"] = (
        avg_time_between_login_attempts_df["avg_time_between_login_attempts"].fillna(0)
    )
else:
    avg_time_between_login_attempts_df = pd.DataFrame(
        columns=["ip", "avg_time_between_login_attempts"]
    )


# =====================================================
# 14. Build final feature dataset
# =====================================================
features_df = request_count_df.copy()

features_df = features_df.merge(max_req_count_5min_df, on="ip", how="left")
features_df = features_df.merge(avg_req_count_5min_df, on="ip", how="left")
features_df = features_df.merge(requests_per_second_df, on="ip", how="left")
features_df = features_df.merge(avg_time_between_requests_df, on="ip", how="left")
features_df = features_df.merge(unique_endpoints_df, on="ip", how="left")
features_df = features_df.merge(unique_ids_df, on="ip", how="left")
features_df = features_df.merge(status_404_df, on="ip", how="left")
features_df = features_df.merge(max_error_rate_5min_df, on="ip", how="left")
features_df = features_df.merge(suspicious_ua_df, on="ip", how="left")
features_df = features_df.merge(bot_ratio_df, on="ip", how="left")
features_df = features_df.merge(mobile_ratio_df, on="ip", how="left")
features_df = features_df.merge(post_frequency_df, on="ip", how="left")
features_df = features_df.merge(max_repeated_endpoint_hits_df, on="ip", how="left")
features_df = features_df.merge(login_attempt_count_df, on="ip", how="left")
features_df = features_df.merge(failed_login_count_df, on="ip", how="left")
features_df = features_df.merge(failed_login_rate_df, on="ip", how="left")
features_df = features_df.merge(max_login_req_per_min_df, on="ip", how="left")
features_df = features_df.merge(avg_time_between_login_attempts_df, on="ip", how="left")

# =====================================================
# 15. Clean final dataset
# =====================================================
features_df = features_df.fillna(0)


# =====================================================
# 16. Display final dataset
# =====================================================
features_df.head()

,ip,request_count,max_req_count_5min,avg_req_count_5min,requests_per_second,avg_time_between_requests,unique_endpoints,unique_ids_accessed,status_404_count,max_error_rate_5min,suspicious_ua_ratio,bot_ratio,mobile_ratio,post_frequency,max_repeated_endpoint_hits,login_attempt_count,failed_login_count,failed_login_rate,max_login_req_per_min,avg_time_between_login_attempts
0,10.124.2.223,2,2,2.0,0.666667,3.0,2,5,0,0.0,0.0,0.0,1.0,0.0,1,0,0,0.0,0.0,0.0
1,102.102.124.56,1,1,1.0,1.000000,0.0,1,3,0,0.0,0.0,0.0,0.0,0.0,1,0,0,0.0,0.0,0.0
2,103.102.221.226,1,1,1.0,1.000000,0.0,1,1,0,0.0,0.0,0.0,0.0,0.0,1,0,0,0.0,0.0,0.0
3,103.216.160.242,2,2,2.0,1.000000,2.0,2,10,0,0.0,0.0,0.0,1.0,0.0,1,0,0,0.0,0.0,0.0
4,103.28.132.72,1,1,1.0,1.000000,0.0,1,4,0,0.0,0.0,0.0,1.0,0.0,1,0,0,0.0,0.0,0.0


In [21]:
features_df.to_csv(
    project_path + "data/processed/features_dataset.csv",
    index=False
)

In [22]:
features_df.isnull().sum()

,0
ip,0
request_count,0
max_req_count_5min,0
avg_req_count_5min,0
requests_per_second,0
avg_time_between_requests,0
unique_endpoints,0
unique_ids_accessed,0
status_404_count,0
max_error_rate_5min,0


In [23]:
features_df.duplicated().sum()

np.int64(0)

In [24]:
features_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2173 entries, 0 to 2172
Data columns (total 20 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   ip                               2173 non-null   object 
 1   request_count                    2173 non-null   int64  
 2   max_req_count_5min               2173 non-null   int64  
 3   avg_req_count_5min               2173 non-null   float64
 4   requests_per_second              2173 non-null   float64
 5   avg_time_between_requests        2173 non-null   float64
 6   unique_endpoints                 2173 non-null   int64  
 7   unique_ids_accessed              2173 non-null   int64  
 8   status_404_count                 2173 non-null   int64  
 9   max_error_rate_5min              2173 non-null   float64
 10  suspicious_ua_ratio              2173 non-null   float64
 11  bot_ratio                        2173 non-null   float64
 12  mobile_ratio        

In [25]:
X = features_df.drop(columns=["ip"])

In [26]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled[:5]

array([[-0.10943373, -0.31382567, -0.30641703, -0.03468386, -0.39315585,
        -0.11058318, -0.16292244, -0.04288762, -0.1960198 , -0.51677403,
        -0.51445011,  0.76123527, -0.05597187, -0.08245719, -0.0629781 ,
         0.        ,  0.        , -0.10376571, -0.04685715],
       [-0.11470528, -0.35654859, -0.36197848,  0.17651019, -0.39618173,
        -0.11669481, -0.18095439, -0.04288762, -0.1960198 , -0.51677403,
        -0.51445011, -1.32317073, -0.05597187, -0.08245719, -0.0629781 ,
         0.        ,  0.        , -0.10376571, -0.04685715],
       [-0.11470528, -0.35654859, -0.36197848,  0.17651019, -0.39618173,
        -0.11669481, -0.19898635, -0.04288762, -0.1960198 , -0.51677403,
        -0.51445011, -1.32317073, -0.05597187, -0.08245719, -0.0629781 ,
         0.        ,  0.        , -0.10376571, -0.04685715],
       [-0.10943373, -0.31382567, -0.30641703,  0.17651019, -0.39416448,
        -0.11058318, -0.11784254, -0.04288762, -0.1960198 , -0.51677403,
        -0.514

In [27]:
joblib.dump(
    scaler,
    project_path + "models/scaler.pkl"
)

['/content/drive/MyDrive/Mobile_API_Misuse_Detector/models/scaler.pkl']